In [ ]:
from pathlib import Path
import sys
import time

sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from ultralytics import YOLO
import torch
import src.config as cfg
from src.training_logger import TrainingLogger

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
ROOT = Path.cwd().resolve().parent.parent
PRETRAINED_DIR = ROOT / "pretrained_models"

logger = TrainingLogger(log_dir=str(ROOT / "training_logs"))

MODEL_SIZE = "s"
EPOCHS = 100
BATCH_SIZE = 16
IMGSZ = 768

In [ ]:
model = YOLO(str(PRETRAINED_DIR / f"yolo26{MODEL_SIZE}.pt"))

start = time.time()

results = model.train(
    data=str(cfg.DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
)

train_time = time.time() - start

In [ ]:
metrics = model.val(workers=0)

baseline_map50, baseline_map50_95 = 0.9262, 0.7663

print(f"mAP50:    {metrics.box.map50:.4f}  ({metrics.box.map50 - baseline_map50:+.4f})")
print(f"mAP50-95: {metrics.box.map:.4f}  ({metrics.box.map - baseline_map50_95:+.4f})")

In [ ]:
train_id, _ = logger.log_training(
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    imgsz=IMGSZ,
    map50=metrics.box.map50,
    map50_95=metrics.box.map,
    train_time=train_time,
    notes="",
)

logger.summary()